# Dataset Builder — Sahih al-Bukhari (Sunnah.com)

Notebook ini membuat dataset hadis Sahih al-Bukhari dari sumber publik
Sunnah.com, lalu menerjemahkan terjemahan Inggris ke Indonesia dengan
`Helsinki-NLP/opus-mt-en-id`.

Notebook ini **bukan** pipeline summarization utama. Notebook ini hanya
menghasilkan dataset (`dataset_hadis_minimal.csv`) yang nanti bisa dipakai
oleh pipeline utama proyek (`run_pipeline.py` / `run_light_pipeline.py`).

Default aman: hanya mengambil sekitar 10 hadis pertama dari 1 book. Ubah
`RUN_FULL_SCRAPE = True` secara manual untuk mengambil seluruh Sahih
al-Bukhari.

Dataset ini merupakan dataset turunan dari sumber publik Sunnah.com pada
koleksi Sahih al-Bukhari. Gunakan untuk kepentingan akademik/UAS dan periksa
kembali ketentuan penggunaan sumber asli sebelum menyebarluaskan hasil
olahan.


## 1. Install Dependency

In [ ]:
# torch dan transformers biasanya sudah tersedia di runtime Colab.
# Paket lain untuk dataset builder dipasang di sini.
!pip install -q beautifulsoup4 requests pandas tqdm tenacity sentencepiece accelerate openpyxl


## 2. Import Library

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


## 3. Upload `create_hadith_dataset.py`

Notebook ini memakai ulang fungsi-fungsi dari `create_hadith_dataset.py`
(script yang sama dipakai di laptop/PyCharm) supaya logika scraping,
parsing, validasi, dan translation tidak diduplikasi/berbeda antara laptop
dan Colab.

Upload file `create_hadith_dataset.py` dari repo lokal ketika diminta.


In [ ]:
from google.colab import files

uploaded = files.upload()  # pilih create_hadith_dataset.py
assert "create_hadith_dataset.py" in uploaded, "Upload create_hadith_dataset.py terlebih dahulu."

sys.path.insert(0, str(Path.cwd()))
import create_hadith_dataset as hadith_builder


## 4. Konfigurasi Aman

In [ ]:
# Path Colab TIDAK memakai /content secara hardcode di sini; path tetap
# relatif terhadap direktori kerja notebook (lihat bagian setup folder).
USE_GOOGLE_DRIVE = False  # ubah True hanya jika ingin menyimpan hasil ke Drive

hadith_builder.RUN_FULL_SCRAPE = False
hadith_builder.LIMIT_BOOKS = 1
hadith_builder.LIMIT_HADITHS = 10
hadith_builder.RUN_TRANSLATION = True
hadith_builder.SAVE_EVERY = 10
hadith_builder.REQUEST_DELAY_SECONDS = 2
hadith_builder.TIMEOUT_SECONDS = 20
hadith_builder.MAX_RETRIES = 3
hadith_builder.COPY_MINIMAL_TO_INPUT = False

print("RUN_FULL_SCRAPE:", hadith_builder.RUN_FULL_SCRAPE)
print("LIMIT_BOOKS:", hadith_builder.LIMIT_BOOKS)
print("LIMIT_HADITHS:", hadith_builder.LIMIT_HADITHS)
print("RUN_TRANSLATION:", hadith_builder.RUN_TRANSLATION)


### (Opsional) Mount Google Drive

Hanya dijalankan jika `USE_GOOGLE_DRIVE = True`. Defaultnya `False` — hasil
disimpan ke direktori runtime Colab dan bisa didownload manual.

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_output_dir = Path("/content/drive/MyDrive/hadith_dataset_builder")
    drive_output_dir.mkdir(parents=True, exist_ok=True)

    hadith_builder.PROJECT_ROOT = drive_output_dir
    hadith_builder.CACHE_HTML_DIR = drive_output_dir / "data" / "cache" / "html"
    hadith_builder.CACHE_JSON_DIR = drive_output_dir / "data" / "cache" / "json"
    hadith_builder.OUTPUT_DIR = drive_output_dir / "data" / "output"
    hadith_builder.INPUT_DIR = drive_output_dir / "data" / "input"
    hadith_builder.OUTPUT_FULL_PATH = hadith_builder.OUTPUT_DIR / "dataset_hadis_full.csv"
    hadith_builder.OUTPUT_MINIMAL_PATH = hadith_builder.OUTPUT_DIR / "dataset_hadis_minimal.csv"
    hadith_builder.OUTPUT_REF_TEMPLATE_PATH = hadith_builder.OUTPUT_DIR / "ringkasan_referensi_template.csv"
    hadith_builder.OUTPUT_PROGRESS_PATH = hadith_builder.OUTPUT_DIR / "scrape_progress.csv"
    hadith_builder.TRANSLATION_CHECKPOINT_PATH = hadith_builder.OUTPUT_DIR / "dataset_hadis_full.partial.csv"
    hadith_builder.INPUT_MINIMAL_COPY_PATH = hadith_builder.INPUT_DIR / "dataset_hadis.csv"
    print("Menyimpan hasil ke Google Drive:", drive_output_dir)
else:
    print("USE_GOOGLE_DRIVE=False. Hasil disimpan ke runtime Colab (data/ di direktori kerja).")


## 5. Setup Folder

In [ ]:
hadith_builder.setup_logging()
hadith_builder.setup_directories()
print("Folder cache/output siap di:", hadith_builder.OUTPUT_DIR.parent)


## 6. Scraping dengan Cache

Mengambil daftar book dari halaman indeks, lalu mengambil hadis dari setiap
book (dibatasi oleh `LIMIT_BOOKS`/`LIMIT_HADITHS` selama
`RUN_FULL_SCRAPE=False`). HTML mentah dan hasil parsing per book disimpan ke
cache supaya bisa diresume tanpa request ulang.

In [ ]:
import requests
from tqdm.auto import tqdm

session = requests.Session()
session.headers.update({"User-Agent": hadith_builder.USER_AGENT})

index_html = hadith_builder.fetch_url_with_cache(
    session, hadith_builder.COLLECTION_INDEX_URL, "index"
)
assert index_html is not None, "Gagal mengambil halaman indeks Sunnah.com."

all_books = hadith_builder.parse_bukhari_book_list(index_html)
print("Jumlah book ditemukan:", len(all_books))

books_to_process = (
    all_books if hadith_builder.RUN_FULL_SCRAPE else all_books[: hadith_builder.LIMIT_BOOKS]
)

all_records = []
progress_entries = []

for book in tqdm(books_to_process, desc="Books"):
    book_number = book["book_number"]
    book_title = book["book_title"]
    book_url = book["book_url"]
    cache_name = f"book_{book_number}"

    cached_records = hadith_builder.load_book_records_cache(cache_name)
    if cached_records is not None:
        records = cached_records
    else:
        html = hadith_builder.fetch_url_with_cache(session, book_url, cache_name)
        if html is None:
            continue
        records = hadith_builder.parse_bukhari_book_page(html, book_number, book_title)
        hadith_builder.save_book_records_cache(cache_name, records)

    all_records.extend(records)
    if not hadith_builder.RUN_FULL_SCRAPE and len(all_records) >= hadith_builder.LIMIT_HADITHS:
        break

if not hadith_builder.RUN_FULL_SCRAPE:
    all_records = all_records[: hadith_builder.LIMIT_HADITHS]

print("Total hadis mentah terkumpul:", len(all_records))
df = pd.DataFrame(all_records)
df.head()


## 7. Validasi Data

In [ ]:
df = hadith_builder.validate_scraped_dataframe(df)
df["data_status"].value_counts()


## 8. Translation English -> Indonesia

In [ ]:
df = hadith_builder.translate_english_to_indonesian(df)
df[["source_ref", "english_translation", "terjemahan", "translation_status"]].head()


## 9. Simpan Dataset Full dan Minimal

In [ ]:
hadith_builder.save_outputs(df)
print("Full   :", hadith_builder.OUTPUT_FULL_PATH)
print("Minimal:", hadith_builder.OUTPUT_MINIMAL_PATH)


## 10. Template Ringkasan Referensi

Template `ringkasan_referensi_template.csv` sudah otomatis dibuat oleh
`save_outputs()` pada langkah sebelumnya. Kolom `ringkasan_referensi`
sengaja dikosongkan dan perlu diisi/divalidasi manusia sebelum dipakai
sebagai referensi evaluasi ROUGE/BERTScore.

In [ ]:
ref_template = pd.read_csv(hadith_builder.OUTPUT_REF_TEMPLATE_PATH, encoding="utf-8-sig")
ref_template.head()


## 11. Statistik Hasil

In [ ]:
total_hadith = len(df)
total_books = df["book_number"].nunique()
valid_count = int((df["data_status"] == "valid").sum())
problem_count = total_hadith - valid_count
empty_arabic = int((df["teks_arab"].astype(str).str.strip() == "").sum())
empty_english = int((df["english_translation"].astype(str).str.strip() == "").sum())
empty_indo = int((df["terjemahan"].astype(str).str.strip() == "").sum())

print("Total hadis      :", total_hadith)
print("Total book       :", total_books)
print("Data valid       :", valid_count)
print("Data bermasalah  :", problem_count)
print("Arab kosong      :", empty_arabic)
print("English kosong   :", empty_english)
print("Terjemahan kosong:", empty_indo)

df[["Num_hadith" if "Num_hadith" in df.columns else "hadith_number", "teks_arab", "terjemahan"]].head()


## Langkah Selanjutnya

1. Download `data/output/dataset_hadis_minimal.csv` (atau file di Google
   Drive jika `USE_GOOGLE_DRIVE=True`).
2. Salin/rename manual menjadi `data/input/dataset_hadis.csv` di proyek
   lokal.
3. Jalankan pipeline utama seperti biasa, misalnya `python run_light_pipeline.py`.
4. Isi `ringkasan_referensi_template.csv` secara manual sebelum dipakai
   untuk evaluasi ROUGE/BERTScore.

Ingat: hasil ringkasan AI pada pipeline utama bukan tafsir, syarah, atau
fatwa, dan terjemahan Indonesia pada dataset ini adalah hasil mesin yang
perlu diperiksa ulang sebelum dipakai di luar keperluan eksperimen NLP.